# SEG301 | Text Preprocessing and Search Engine Lab (9 bước)
**Đề tài:** FNAC-HNM Vietnamese Dense Retrieval • Auxiliary corpus của 5 website.

Notebook bám thứ tự lab của thầy: **1. Explore → 2. Cleaning → 3. Tokenization → 4. Stopwords → 5. Stemming/Lemma → 6. TF-IDF/BM25 → 7. Strategies → 8. Report → 9. Discussion**.

**Quy ước khoa học:** website crawl là corpus *không có nhãn relevance*; để đánh giá P@5/Recall@5 đúng cần nhãn query-document. Không tự bịa nhãn từ `source_site` hoặc scores. Dùng 4 chiến lược tiền xử lý cho bài lab; để **dense training** vẫn giữ `context` tiếng Việt tự nhiên, không strip stopwords/stem.

## 0. Chuẩn bị môi trường và đường dẫn
Mở thư mục `Preprocessing` bằng VS Code/Jupyter. Trên Windows chạy một lần trong Terminal:

```powershell
python -m pip install -r requirements-preprocessing.txt
python -m nltk.downloader wordnet omw-1.4
```

Notebook tự nhận thư mục Crawler nằm ngang hàng. Nếu preprocessing CSV đã có thì **dùng lại**, không chạy lại SQL không cần thiết.

In [1]:
from pathlib import Path
import json, sys, re
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse

ROOT = Path.cwd().resolve()
if not (ROOT/'preprocess_merged.py').is_file():
    raise RuntimeError(f'Working directory phải là thư mục Preprocessing, hiện tại: {ROOT}')
sys.path.insert(0, str(ROOT))
from core.retrieval_lab import run_lab, search_saved_index, VietnameseTokenizer, _english_tools
from complete_lab import finish_lab

OUT_DIR = ROOT/'data'/'preprocessing'
DB_PATH = ROOT.parent/'Crawler'/'data'/'vire_web_corpus_merged.db'
TOKENIZER = 'underthesea'  # đồng nhất tokenizer với file kết quả đã tạo
LABELS_REVIEWED = ROOT/'qrels_reviewed.csv'  # optional pooled labels; crawl_eval nằm ở thư mục khác

if not (OUT_DIR/'04_crawled_auxiliary_corpus.csv').exists():
    if not DB_PATH.exists():
        raise FileNotFoundError(f'Không tìm thấy DB: {DB_PATH}')
    print('Chưa có CSV, đang chạy preprocessing từ:', DB_PATH)
    run_lab(DB_PATH, OUT_DIR, tokenizer_backend=TOKENIZER)
else:
    print('Đã có dataset preprocessing; dùng lại dữ liệu đã xử lý.')
print('Input DB để tái tạo từ đầu: ../Crawler/data/vire_web_corpus_merged.db')
print('DB hiện diện:', DB_PATH.exists(), '(không bắt buộc nếu CSV đã có)')
print('Thư mục kết quả: data/preprocessing/')
if (OUT_DIR/'report.json').exists():
    print('Tokenizer thực sự của corpus:', json.loads((OUT_DIR/'report.json').read_text(encoding='utf-8')).get('tokenizer_backend'))
if not (OUT_DIR/'QUALITY_GATE.md').exists():
    raise RuntimeError('Chưa chạy phiên bản source-aware cleaning mới! Chạy lại preprocess_merged.py trước.')


Đã có dataset preprocessing; dùng lại dữ liệu đã xử lý.
Input DB để tái tạo từ đầu: ../Crawler/data/vire_web_corpus_merged.db
DB hiện diện: True (không bắt buộc nếu CSV đã có)
Thư mục kết quả: data/preprocessing/
Tokenizer thực sự của corpus: underthesea


## Bước 1. Load & Explore (rows, columns, missing, duplicates, categories)
CSV `01` được trích trực tiếp từ SQLite merge; `04` là tài liệu đã lọc. `source_site` tương tự thông tin loại nguồn, **không phải nhãn relevance**. Loại trùng là hợp lý với chỉ mục tìm kiếm nhưng giữ nguồn gốc URL.

In [2]:
raw = pd.read_csv(OUT_DIR/'01_export_merged_raw.csv', keep_default_na=False)
aux = pd.read_csv(OUT_DIR/'04_crawled_auxiliary_corpus.csv', keep_default_na=False)
lab = pd.read_csv(OUT_DIR/'03_cleaned_tokenized_lab.csv', keep_default_na=False)
print('Raw shape (rows, cols):', raw.shape)
print('Clean shape:', aux.shape)
print('Missing raw text:', raw.raw_text.eq('').sum())
print('Duplicated raw_text:', raw.raw_text.duplicated().sum())
print('Duplicated cleaned context:', aux.context.duplicated().sum())
print('Per source_site:')
display(aux.groupby('source_site').size().rename('count').reset_index())
display(raw.head(3)[['source_site','title','url','raw_text']])

Raw shape (rows, cols): (1022, 12)
Clean shape: (889, 13)
Missing raw text: 0
Duplicated raw_text: 6
Duplicated cleaned context: 0
Per source_site:


,source_site,count
0,fpt_telecom_support,88
1,hcmus_admissions,395
2,mobifone_5g_faq,12
3,momo_help,384
4,uel_admissions_faq,10


,source_site,title,url,raw_text
0,fpt_telecom_support,Làm sao để đăng ký dịch vụ Internet trên Hi FPT?,https://fpt.vn/hi-fpt/huong-dan-su-dung/dang-k...,Làm sao để đăng ký dịch vụ Internet trên Hi FP...
1,fpt_telecom_support,Tôi muốn đăng ký gói Combo Internet Thể Thao t...,https://fpt.vn/hi-fpt/huong-dan-su-dung/dang-k...,Tôi muốn đăng ký gói Combo Internet Thể Thao t...
2,fpt_telecom_support,Tôi muốn đăng ký gói Ultra Fast thì phải làm sao?,https://fpt.vn/hi-fpt/huong-dan-su-dung/dang-k...,Tôi muốn đăng ký gói Ultra Fast thì phải làm s...


## Bước 2. Text Cleaning and Normalization: source-aware + quality gate

Dataset phục vụ nghiên cứu nên cleaning phải **bám cấu trúc từng website**. Quy trình mới: loại đoạn đuôi lặp do crawler → trích xuất nội dung chính từng nguồn → phân tách cặp FAQ khi cần → chia đoạn theo câu → kiểm tra trùng → chạy quality gate.

- **MoMo:** bỏ menu danh mục, đánh giá và bài viết liên quan; chỉ giữ bài hỏi–đáp có cấu trúc nhận diện được.
- **UEL:** tách từng cặp câu hỏi–trả lời thay vì cắt ngang danh sách hỏi đáp.
- **MobiFone:** chỉ dùng trang FAQ đúng chủ đề, không đưa trang marketing vào corpus.
- **FPT/HCMUS:** xóa footer hoặc link điều hướng sau bài, giữ nguyên mức phí, điều kiện, phủ định, URL có ý nghĩa.

Các hàng không chắc chắn được lưu trong `16_quarantine_pages.csv` để xem lại. **0 lỗi theo các mẫu kiểm tra tự động không đảm bảo 100% sạch**. Cần xác nhận bằng mẫu kiểm tra thủ công.


In [3]:
from core.source_cleaning import RESIDUAL_PATTERNS
examples = pd.read_csv(OUT_DIR/'06_raw_clean_examples.csv', keep_default_na=False)
print('Ví dụ trước/sau từ 5 website (KHÔNG tính lại bằng hàm cleaning cũ):')
for source, g in examples.groupby('source_site'):
    row = g.iloc[0]
    print(f'\nNguồn: {source} | rule: {row.extraction_rule}')
    print('RAW  :', str(row.raw_text)[:360])
    print('CLEAN:', str(row.clean_text)[:360])
    print('Tokens:', row.raw_token_count, '→', row.clean_token_count)
qc = pd.read_csv(OUT_DIR/'17_residual_noise_audit.csv', keep_default_na=False)
print('\nSố lỗi khớp các mẫu nhiễu đặc trưng:', int(qc.violations.sum()))
assert qc.violations.sum() == 0, 'Vẫn còn mẫu noise được phát hiện; không nên index.'


Ví dụ trước/sau từ 5 website (KHÔNG tính lại bằng hàm cleaning cũ):

Nguồn: fpt_telecom_support | rule: fpt_footer_trim
RAW  : Làm sao để đăng ký dịch vụ Internet trên Hi FPT?
Quý Khách có thể chủ động đăng ký Internet FPT trên Hi FPT theo hướng dẫn sau: Bước 1: Nhấn chọn "Internet" tại bảng "Đăng ký dịch vụ" dưới màn hình trang chủ Bước 2: Chọn gói cước tại mục gợi ý "Bán chạy" hoặc tại mục "Internet", nhấn "Mua ngay" Bước 3: Tích chọn địa chỉ lắp đặt cũ hoặc "Thêm địa chỉ mới" (nế
CLEAN: Làm sao để đăng ký dịch vụ Internet trên Hi FPT? Quý Khách có thể chủ động đăng ký Internet FPT trên Hi FPT theo hướng dẫn sau: Bước 1: Nhấn chọn "Internet" tại bảng "Đăng ký dịch vụ" dưới màn hình trang chủ Bước 2: Chọn gói cước tại mục gợi ý "Bán chạy" hoặc tại mục "Internet", nhấn "Mua ngay" Bước 3: Tích chọn địa chỉ lắp đặt cũ hoặc "Thêm địa chỉ mới" (nế
Tokens: 124 → 117

Nguồn: hcmus_admissions | rule: hcmus_article_trim
RAW  : THÔNG BÁO VỀ PHƯƠNG THỨC XÉT TUYỂN 1 NĂM 2024
Hội đồng tuyển sinh T

### Audit dữ liệu bị loại & kiểm tra thủ công

Xem `15_cleaning_page_audit.csv` để truy vết quyết định theo URL; `16_quarantine_pages.csv` là những đoạn cần giữ để điều tra, **không nằm trong corpus train**. `18_manual_review_sample.csv` là mẫu ngẫu nhiên cố định từ năm website; phải tự đọc nội dung và chấm chất lượng trước khi chốt nghiên cứu. Báo cáo `QUALITY_GATE.md` nêu rõ trạng thái PASS tự động và PENDING kiểm tra bằng người.


In [4]:
audit=pd.read_csv(OUT_DIR/'15_cleaning_page_audit.csv',keep_default_na=False)
quarantine=pd.read_csv(OUT_DIR/'16_quarantine_pages.csv',keep_default_na=False)
review=pd.read_csv(OUT_DIR/'18_manual_review_sample.csv',keep_default_na=False)
print('Các quyết định trích xuất:')
display(audit.groupby(['source_site','action']).size().rename('passages').reset_index())
print('Lý do quarantine (các nhóm phổ biến):')
display(quarantine.reason.value_counts().rename_axis('reason').reset_index(name='count').head(12))
print('Mẫu đầu ra cần review ở năm nguồn:')
display(review.groupby('source_site').head(1)[['source_site','title','context','url']])
flags=pd.read_csv(OUT_DIR/'19_review_flags.csv',keep_default_na=False)
form=pd.read_csv(OUT_DIR/'20_review_form_TEMPLATE.csv',keep_default_na=False)
print('Đoạn cần kiểm tra kỹ thêm:',len(flags),'| biểu mẫu kiểm tra:',len(form))
display(flags[['flag','source_site','title','context']].head(4))
print('Chi tiết: QUALITY_GATE.md; điền form thành review_decisions.csv rồi chạy --reviews để loại bản ghi khi cần')

# Tập dùng cho nghiên cứu: các bản ghi cần review được giữ riêng, không âm thầm hòa vào mining pool.
ready = pd.read_csv(OUT_DIR/'24_auxiliary_corpus_RESEARCH_READY.csv',keep_default_na=False)
pending = pd.read_csv(OUT_DIR/'25_pending_manual_review.csv',keep_default_na=False)
print('Full lab corpus:',len(aux),'| Research-ready:',len(ready),'| Pending human review:',len(pending))
print('Research-ready by source:')
display(ready.groupby('source_site').size().rename('documents').reset_index())


Các quyết định trích xuất:


,source_site,action,passages
0,fpt_telecom_support,extracted,88
1,hcmus_admissions,extracted,40
2,hcmus_admissions,quarantine,41
3,mobifone_5g_faq,extracted,1
4,mobifone_5g_faq,quarantine,26
5,momo_help,extracted,336
6,momo_help,quarantine,464
7,uel_admissions_faq,extracted,14
8,uel_admissions_faq,quarantine,12


Lý do quarantine (các nhóm phổ biến):


,reason,count
0,secondary_duplicate_tail_passage,463
1,momo_category_listing_without_answer,61
2,mobi_non_faq_marketing_or_category_page,13
3,momo_answer_too_short_or_empty,2
4,hcmus_landing_navigation,1
5,hcmus_admission_method_directory_navigation,1
6,momo_listing_home,1
7,uel_no_safe_qa_cards,1


Mẫu đầu ra cần review ở năm nguồn:


,source_site,title,context,url
0,fpt_telecom_support,Làm sao để đăng ký dịch vụ Internet trên Hi FPT?,Làm sao để đăng ký dịch vụ Internet trên Hi FP...,https://fpt.vn/hi-fpt/huong-dan-su-dung/dang-k...
12,hcmus_admissions,THÔNG BÁO VỀ PHƯƠNG THỨC XÉT TUYỂN 1 NĂM 2024,THÔNG BÁO VỀ PHƯƠNG THỨC XÉT TUYỂN 1 NĂM 2024 ...,https://tuyensinh.hcmus.edu.vn/2024-thong-bao-...
25,mobifone_5g_faq,Mạng 5G là gì?,Mạng 5G là gì? 5G là thế hệ mạng thứ năm của c...,https://5g.mobifone.vn/mobifone5g/danh-muc-chi...
35,momo_help,Ai có thể gây quỹ trên Trái Tim MoMo?,Ai có thể gây quỹ trên Trái Tim MoMo? Để có th...,https://www.momo.vn/hoi-dap/ai-co-the-gay-quy-...
48,uel_admissions_faq,Trường Đại học Kinh tế – Luật có bao nhiêu cơ ...,Trường Đại học Kinh tế – Luật có bao nhiêu cơ ...,https://tuyensinh.uel.edu.vn/faq/


Đoạn cần kiểm tra kỹ thêm: 2 | biểu mẫu kiểm tra: 42


,flag,source_site,title,context
0,hcmus_program_code_discrepancy_between_pages,hcmus_admissions,Công nghệ Vật lý điện tử và tin học (Chương tr...,Chỉ tiêu tuyển sinh 2026 | Mã ngành: 7440107 |...
1,hcmus_program_code_discrepancy_between_pages,hcmus_admissions,Công nghệ Vật lý điện tử và tin học (Chương tr...,Chỉ tiêu tuyển sinh 2026 | Mã ngành: 7440107_D...


Chi tiết: QUALITY_GATE.md; điền form thành review_decisions.csv rồi chạy --reviews để loại bản ghi khi cần
Full lab corpus: 889 | Research-ready: 887 | Pending human review: 2
Research-ready by source:


,source_site,documents
0,fpt_telecom_support,88
1,hcmus_admissions,393
2,mobifone_5g_faq,12
3,momo_help,384
4,uel_admissions_faq,10


### Kiểm định HCMUS 2026 theo DB gốc

Parser đã sửa từ chính `vire_web_corpus_merged.db`: cả **hai bảng tuyển sinh đều đủ 39/39 STT**, gồm mã chữ-số `74401a1` (Công nghệ bán dẫn) và `75202a1` (Thiết kế vi mạch). Bước kiểm định từ CSV nguồn bảo đảm mỗi hàng có mã ngành, chỉ tiêu và tên ngành riêng, không dính hàng kế tiếp.

- `21_hcmus_table_integrity.csv`: 39 hàng trên mỗi trang bảng và STT bị thiếu, nếu có.
- `22_hcmus_chunk_integrity.csv`: kiểm tra chunk không cắt lửng và hàng dài.
- `23_hcmus_code_discrepancies.csv`: khác biệt mã ngành giữa hai trang vẫn để riêng để đối chiếu, **không tự sửa**.

Chưa thể xác nhận mọi thông tin tuyển sinh đúng thực tế chỉ từ DB; muốn kiểm chứng tuyệt đối phải đối chiếu thông báo gốc.

In [5]:
hcmus_table = pd.read_csv(OUT_DIR/'21_hcmus_table_integrity.csv', keep_default_na=False)
hcmus_chunk = pd.read_csv(OUT_DIR/'22_hcmus_chunk_integrity.csv', keep_default_na=False)
hcmus_conflicts = pd.read_csv(OUT_DIR/'23_hcmus_code_discrepancies.csv', keep_default_na=False)
print('HCMUS 2026: kiểm định bảng nguồn')
display(hcmus_table)
print('Ranh giới đứt mục:', int(hcmus_chunk.dangling_number.sum()))
print('Hàng vượt giới hạn số từ:', int(hcmus_chunk.long_atomic_row.sum()))
print('Đối chiếu mã ngành:', len(hcmus_conflicts))
display(hcmus_conflicts)
print('Các đơn vị bảng chỉ tiêu (ví dụ)')
display(aux.loc[(aux.source_site=='hcmus_admissions') & (aux.extraction_rule=='hcmus_structured_admission_row'),['title','url','context']].head(3))
assert len(hcmus_table)==2 and (hcmus_table.parsed_rows==39).all(), 'Không đủ 39 hàng ở hai bảng HCMUS!'
assert (hcmus_table.missing_stt.fillna('').astype(str)=='').all(), 'Còn STT thiếu!'
print('Đã xác nhận đủ 39/39 mã ngành ở cả 2 bảng HCMUS.')


HCMUS 2026: kiểm định bảng nguồn


,url,status,parsed_rows,missing_stt,warning
0,https://tuyensinh.hcmus.edu.vn/2026-phuong-huo...,parsed_complete_39_rows,39,,
1,https://tuyensinh.hcmus.edu.vn/chi-tieu/,parsed_complete_39_rows,39,,


Ranh giới đứt mục: 0
Hàng vượt giới hạn số từ: 0
Đối chiếu mã ngành: 1


,program,quota,codes,urls,status
0,công nghệ vật lý điện tử và tin học (chương tr...,60,7440107 | 7440107_DKD,https://tuyensinh.hcmus.edu.vn/2026-phuong-huo...,MANUAL_REVIEW_REQUIRED_NOT_AUTOCORRECTED


Các đơn vị bảng chỉ tiêu (ví dụ)


,title,url,context
230,"Sinh học (mã ngành 7420101, chỉ tiêu 215, 2026)",https://tuyensinh.hcmus.edu.vn/2026-phuong-huo...,Chỉ tiêu tuyển sinh 2026 | Mã ngành: 7420101 |...
231,Sinh học (Chương trình tăng cường tiếng Anh) (...,https://tuyensinh.hcmus.edu.vn/2026-phuong-huo...,Chỉ tiêu tuyển sinh 2026 | Mã ngành: 7420101_D...
232,"Công nghệ Sinh học (mã ngành 7420201, chỉ tiêu...",https://tuyensinh.hcmus.edu.vn/2026-phuong-huo...,Chỉ tiêu tuyển sinh 2026 | Mã ngành: 7420201 |...


Đã xác nhận đủ 39/39 mã ngành ở cả 2 bảng HCMUS.


## Bước 3. Vietnamese Tokenization + Token Counts
Đề bài dùng NLTK cho tiếng Anh; nhóm dùng `underthesea` để giữ từ ghép như `chăm_sóc`, `khách_hàng`. Không phải embedding, chỉ là danh sách từ phục vụ TF-IDF/BM25.

So token counts raw/clean **cùng tokenizer**; file `03` lưu số token cho toàn corpus.

In [6]:
for i,row in lab.head(3).iterrows():
    tokens = json.loads(row.tokens_baseline)
    print(f'{i+1}. {row.source_site}:', tokens[:30])
    print('    raw_token_count:', row.raw_token_count, '| clean_token_count:', row.clean_token_count,
          '| baseline:', row.count_baseline)
print('Raw token count mean:',round(lab.raw_token_count.mean(),2))
print('Clean token count mean:',round(lab.clean_token_count.mean(),2))
print(lab[['raw_token_count','clean_token_count']].describe())

1. fpt_telecom_support: ['làm_sao', 'để', 'đăng_ký', 'dịch_vụ', 'internet', 'trên', 'hi_fpt', 'quý_khách', 'có_thể', 'chủ_động', 'đăng_ký', 'internet', 'fpt', 'trên', 'hi_fpt', 'theo', 'hướng_dẫn', 'sau', 'bước', '1', 'nhấn', 'chọn', 'internet', 'tại', 'bảng', 'đăng_ký', 'dịch_vụ', 'dưới', 'màn_hình', 'trang_chủ']
    raw_token_count: 117 | clean_token_count: 117 | baseline: 88
2. fpt_telecom_support: ['tôi', 'muốn', 'đăng_ký', 'gói', 'combo', 'internet', 'thể_thao', 'thì', 'phải', 'làm_sao', 'quý_khách', 'có_thể', 'chủ_động', 'đăng_ký', 'gói', 'combo', 'truyền_hình', 'trên', 'hi_fpt', 'theo', 'hướng_dẫn', 'sau', 'bước', '1', 'nhấn', 'chọn', 'combo', 'tại', 'bảng', 'đăng_ký']
    raw_token_count: 112 | clean_token_count: 112 | baseline: 86
3. fpt_telecom_support: ['tôi', 'muốn', 'đăng_ký', 'gói', 'ultra_fast', 'thì', 'phải', 'làm_sao', 'quý_khách', 'có_thể', 'đăng_ký', 'dịch_vụ', 'ultra', 'fast', 'trên', 'hi_fpt', 'theo', 'hướng_dẫn', 'sau', 'bước', '1', 'nhấn', 'chọn', 'ultra', 'fast'

## Bước 4. Stopword Removal (A vs B) + Token Reduction
- A: loại stopwords trên token list gốc, **giữ dấu câu**.
- B: loại stopwords và chỉ giữ token dạng chữ/số.

Bỏ stopwords không phải lúc nào cũng cải thiện retrieval. Đặc biệt phải giữ `không`, `chưa`, `chẳng`, `đừng` vì đổi nghĩa câu.

In [7]:
base_n = int(lab.count_baseline.sum())
nostop_n = int(lab.count_no_stopwords.sum())
print(f'Baseline tokens: {base_n:,}; No-stopword: {nostop_n:,}; reduction: {(base_n-nostop_n)/base_n*100:.2f}%')
for _,row in lab.head(3).iterrows():
    print('\nBaseline:',json.loads(row.tokens_baseline)[:20])
    print('Exp A   :',json.loads(row.tokens_no_stop_raw)[:20])
    print('Exp B   :',json.loads(row.tokens_no_stop_alnum)[:20])
from core.retrieval_lab import STOPWORDS_VI
print('Words of negation removed?:', {w:(w in STOPWORDS_VI) for w in ('không','chưa','chẳng','đừng')})

Baseline tokens: 93,829; No-stopword: 78,046; reduction: 16.82%

Baseline: ['làm_sao', 'để', 'đăng_ký', 'dịch_vụ', 'internet', 'trên', 'hi_fpt', 'quý_khách', 'có_thể', 'chủ_động', 'đăng_ký', 'internet', 'fpt', 'trên', 'hi_fpt', 'theo', 'hướng_dẫn', 'sau', 'bước', '1']
Exp A   : ['làm_sao', 'đăng_ký', 'dịch_vụ', 'internet', 'hi_fpt', '?', 'quý_khách', 'có_thể', 'chủ_động', 'đăng_ký', 'internet', 'fpt', 'hi_fpt', 'hướng_dẫn', 'sau', ':', 'bước', '1', ':', 'nhấn']
Exp B   : ['làm_sao', 'đăng_ký', 'dịch_vụ', 'internet', 'hi_fpt', 'quý_khách', 'có_thể', 'chủ_động', 'đăng_ký', 'internet', 'fpt', 'hi_fpt', 'hướng_dẫn', 'sau', 'bước', '1', 'nhấn', 'chọn', 'internet', 'bảng']

Baseline: ['tôi', 'muốn', 'đăng_ký', 'gói', 'combo', 'internet', 'thể_thao', 'thì', 'phải', 'làm_sao', 'quý_khách', 'có_thể', 'chủ_động', 'đăng_ký', 'gói', 'combo', 'truyền_hình', 'trên', 'hi_fpt', 'theo']
Exp A   : ['muốn', 'đăng_ký', 'gói', 'combo', 'internet', 'thể_thao', 'phải', 'làm_sao', '?', 'quý_khách', 'có_thể', 

## Bước 5. Stemming vs. Lemmatization
**Lưu ý:** PorterStemmer và WordNet **là của tiếng Anh**. Corpus tiếng Việt không có cùng dạng biến đổi morphological. Do đó `stemmed`/`lemmatized` chỉ áp dụng cho một số từ tiếng Anh minh họa; không được khẳng định đã lemmatize toàn bộ tiếng Việt.

Stemming có thể tạo stem không phải từ hoàn chỉnh; lemma dựa POS và từ điển (`studies → study`, `running → run` nếu POS động từ).

In [8]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
stemmer = PorterStemmer()
words = ['connect','connected','connecting','connection','studies','studying','cars','running','better']
try:
    wnl = WordNetLemmatizer()
    wnl.lemmatize('cars')
    records = [{'word':w,'porter_stem':stemmer.stem(w),
                'lemma_noun':wnl.lemmatize(w,pos='n'),
                'lemma_verb':wnl.lemmatize(w,pos='v'),
                'lemma_adj':wnl.lemmatize(w,pos='a')} for w in words]
    display(pd.DataFrame(records))
except LookupError:
    print('WordNet chưa tải: chạy python -m nltk.downloader wordnet omw-1.4')
    display(pd.DataFrame({'word':words,'porter_stem':[stemmer.stem(w) for w in words]}))
strategies = pd.read_csv(OUT_DIR/'08_strategy_comparison.csv',keep_default_na=False)
display(strategies[['strategy','vocab_size','token_count','token_reduction_pct']])
print('Ghi nhớ: vocabulary stemming/lemmatization trên corpus Việt có thể gần/đúng bằng no_stopwords.')
print('\nBảng minh họa tiếng Anh đã lưu (dữ liệu có sẵn từ bài lab):')
english_demo = pd.read_csv(OUT_DIR/'07_stemming_lemma_english_demo.csv')
display(english_demo)


,word,porter_stem,lemma_noun,lemma_verb,lemma_adj
0,connect,connect,connect,connect,connect
1,connected,connect,connected,connect,connected
2,connecting,connect,connecting,connect,connecting
3,connection,connect,connection,connection,connection
4,studies,studi,study,study,studies
5,studying,studi,studying,study,studying
6,cars,car,car,cars,cars
7,running,run,running,run,running
8,better,better,better,better,good


,strategy,vocab_size,token_count,token_reduction_pct
0,baseline,3193,93829,0.00
1,no_stopwords,3143,78046,16.82
2,stemmed,3143,78046,16.82
3,lemmatized,3143,78046,16.82


Ghi nhớ: vocabulary stemming/lemmatization trên corpus Việt có thể gần/đúng bằng no_stopwords.

Bảng minh họa tiếng Anh đã lưu (dữ liệu có sẵn từ bài lab):


,token,stemmed_english,lemmatized_english
0,connect,connect,connect
1,connected,connect,connect
2,connecting,connect,connect
3,connection,connect,connection
4,studies,studi,study
5,studying,studi,study
6,cars,car,car
7,running,run,run


## Bước 6. Search Engine: TF-IDF + Cosine Similarity and BM25
Cùng dữ liệu, cùng tokenization cho *document và query*. Ma trận TF-IDF sparse lưu bằng NPZ; BM25 dùng term frequency + IDF + độ dài tài liệu.

**Không so độ lớn raw score giữa TF-IDF cosine và BM25**. So thứ hạng và relevance metrics.

In [9]:
X = sparse.load_npz(OUT_DIR/'models'/'tfidf_baseline.npz')
print('Sparse TF-IDF matrix:', X.shape, '| nonzero:',X.nnz)
ranked=pd.read_csv(OUT_DIR/'09_tfidf_bm25_top10.csv',keep_default_na=False)
qids=ranked[['query_id','query']].drop_duplicates()
print('Test queries:',len(qids))
for qid,query in qids.head(5).itertuples(index=False):
    print('\nQUERY:', query)
    for method in ['tfidf','bm25']:
        top = ranked[(ranked.query_id == qid) & (ranked.method == method) & (ranked.strategy == 'baseline') & (ranked['rank'] <= 5)]
        print(method.upper(), list(top.doc_id))

Sparse TF-IDF matrix: (889, 3193) | nonzero: 56484
Test queries: 5

QUERY: Cách kiểm tra gói cước 5G MobiFone đang sử dụng?
TFIDF ['web_dee84d7bc9a61efe2681', 'web_7b834eb290850b150d8c', 'web_36aa797ea6489796b976', 'web_14b5b6a6faff1f9e2c0c', 'web_00836287d6ee815d027e']
BM25 ['web_dee84d7bc9a61efe2681', 'web_7b834eb290850b150d8c', 'web_15efaa166448e4ce2a02', 'web_36aa797ea6489796b976', 'web_14b5b6a6faff1f9e2c0c']

QUERY: Làm sao liên hệ chăm sóc khách hàng MoMo?
TFIDF ['web_643a7599dc7f92ee34d1', 'web_3acc339aecd3f6932d61', 'web_740cd08f60cd570304d7', 'web_5237eb23c95eef6e2f66', 'web_cf08db07ad19bd88b024']
BM25 ['web_643a7599dc7f92ee34d1', 'web_740cd08f60cd570304d7', 'web_3acc339aecd3f6932d61', 'web_5237eb23c95eef6e2f66', 'web_ae2699f078ffa39e9218']

QUERY: Cách thay đổi mật khẩu WiFi FPT?
TFIDF ['web_f26f0a0e97001e39e8a1', 'web_5333fe03c99f41f5c58f', 'web_720969c81122a6a0dcb1', 'web_b4da9351caf3f625d77b', 'web_5e5b3c8bd86c3e3191ed']
BM25 ['web_a592c195028029eacf09', 'web_5e2ed39471583

## Bước 7. Compare Four Preprocessing Strategies
Giữ **ranker cố định** khi so preprocessing; giữ **preprocessing cố định** khi so TF-IDF với BM25. Báo cáo vocabulary size, token reduction, và **Top-5 doc IDs** của 4 strategies × 2 models. Nếu corpus chưa có query relevance thì chỉ kết luận thay đổi ranking, chưa kết luận chất lượng tăng.

In [10]:
display(strategies[['strategy','vocab_size','tfidf_rows','tfidf_cols','token_reduction_pct']])
report = finish_lab(OUT_DIR,
    labels=LABELS_REVIEWED if LABELS_REVIEWED.exists() else None,
    tokenizer_backend=TOKENIZER)
print(json.dumps({k:v for k,v in report.items() if k != 'report'},indent=2,ensure_ascii=False))
print('Báo cáo tổng hợp: data/preprocessing/REPORT_SEG301_COMPLETE.md')
comparison=pd.read_csv(OUT_DIR/'12_top5_comparison.csv',keep_default_na=False)
display(comparison.head(16)[['query_id','strategy','method','overlap_with_baseline','exact_same_order_as_baseline','top5_ids']])

,strategy,vocab_size,tfidf_rows,tfidf_cols,token_reduction_pct
0,baseline,3193,889,3193,0.00
1,no_stopwords,3143,889,3143,16.82
2,stemmed,3143,889,3143,16.82
3,lemmatized,3143,889,3143,16.82


{
  "corpus_documents": 889,
  "raw_passages": 1022,
  "top5_rows": 200,
  "strategy_comparison_rows": 40,
  "manual_p5_scored": null,
  "crawl_metrics_scored": true
}
Báo cáo tổng hợp: data/preprocessing/REPORT_SEG301_COMPLETE.md


,query_id,strategy,method,overlap_with_baseline,exact_same_order_as_baseline,top5_ids
0,q01,baseline,bm25,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
1,q01,no_stopwords,bm25,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
2,q01,stemmed,bm25,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
3,q01,lemmatized,bm25,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
4,q01,baseline,tfidf,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
5,q01,no_stopwords,tfidf,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
6,q01,stemmed,tfidf,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
7,q01,lemmatized,tfidf,5,1,web_dee84d7bc9a61efe2681 | web_7b834eb290850b1...
8,q02,baseline,bm25,5,1,web_643a7599dc7f92ee34d1 | web_740cd08f60cd570...
9,q02,no_stopwords,bm25,4,0,web_643a7599dc7f92ee34d1 | web_3acc339aecd3f69...


## Bước 8. Đánh giá truy xuất có nhãn từ chính 5 website crawl

Tập đánh giá v2 có **50 documents, 40 truy vấn**, tạo **2.000 cặp query–document** được **AI hỗ trợ gán nhãn sơ bộ**, chưa có xác nhận độc lập của con người. Có 47 nhãn `relevant=1`, các truy vấn có từ 1–3 positive. Không dùng CSConDa hoặc EduCoQA trong bài SEG301.

- **Chỉ trên 50 documents gán đủ nhãn:** tính Precision@5, Recall@5, Hit@1, MRR@5 và nDCG@5 (4 chiến lược × 2 rankers).
- **Bổ sung trên toàn bộ 887 documents:** kiểm tra liệu tài liệu *đã biết positive* còn xuất hiện ở Top-1/5/10 hay không. **Không được gọi các số này là Precision/Recall trên 887 tài liệu** vì những documents còn lại chưa có nhãn relevance.
- **Giới hạn:** Câu hỏi được soạn từ corpus, nhãn AI là bước chuẩn bị ban đầu; cần kiểm tra độc lập trước khi coi là benchmark công bố.

Chạy `RUN_EVAL_V2_WINDOWS.bat` hoặc thực hiện ba lệnh trong `HUONG_DAN_EVAL_V2.md`.

In [11]:
from IPython.display import display
benchmark = OUT_DIR/'crawl_eval'
print('KẾT QUẢ ĐÁNH GIÁ TỪ 5 WEBSITE CRAWL')
print('Corpus lab:',len(aux),'documents | Research-ready:',len(ready),'documents')
print('Dữ liệu chính: 01_export_merged_raw.csv, 04_crawled_auxiliary_corpus.csv, 24_auxiliary_corpus_RESEARCH_READY.csv')
for f in ('eval_queries.csv','eval_documents.csv','qrels_REVIEWED.csv','metric_summary.csv'):
    print(' - crawl_eval/'+f+':', 'CÓ' if (benchmark/f).exists() else 'CHƯA CÓ')
if (benchmark/'metric_summary.csv').exists():
    metric_table=pd.read_csv(benchmark/'metric_summary.csv')
    print('Bộ metric đầy đủ của 4 strategies × 2 rankers:')
    display(metric_table.round(4))
else:
    print('Chưa có metric: chạy RUN_EVAL_V2_WINDOWS.bat sau khi chuẩn bị đủ nhãn.')


KẾT QUẢ ĐÁNH GIÁ TỪ 5 WEBSITE CRAWL
Corpus lab: 889 documents | Research-ready: 887 documents
Dữ liệu chính: 01_export_merged_raw.csv, 04_crawled_auxiliary_corpus.csv, 24_auxiliary_corpus_RESEARCH_READY.csv
 - crawl_eval/eval_queries.csv: CÓ
 - crawl_eval/eval_documents.csv: CÓ
 - crawl_eval/qrels_REVIEWED.csv: CÓ
 - crawl_eval/metric_summary.csv: CÓ
Bộ metric đầy đủ của 4 strategies × 2 rankers:


,strategy,method,P_at_5,Recall_at_5,Hit_at_1,MRR_at_5,nDCG_at_5,queries_scored,queries_total
0,baseline,bm25,0.230,0.9917,0.975,0.9875,0.9849,40,40
1,baseline,tfidf,0.225,0.9833,0.925,0.9500,0.9539,40,40
2,lemmatized,bm25,0.230,0.9917,0.950,0.9750,0.9757,40,40
3,lemmatized,tfidf,0.230,0.9917,0.875,0.9271,0.9389,40,40
4,no_stopwords,bm25,0.230,0.9917,0.950,0.9750,0.9757,40,40
5,no_stopwords,tfidf,0.230,0.9917,0.875,0.9271,0.9389,40,40
6,stemmed,bm25,0.230,0.9917,0.950,0.9750,0.9757,40,40
7,stemmed,tfidf,0.230,0.9917,0.875,0.9271,0.9389,40,40


### Bước 8 bổ sung: metrics có nhãn, đánh giá 887 tài liệu và thí nghiệm hình thái học

- `evaluate_crawled_corpus.py evaluate --tokenizer underthesea`: tính P@5, Recall@5, Hit@1, MRR@5 và nDCG@5 trên **50 tài liệu có nhãn AI sơ bộ**.
- `evaluate_887_diagnostic.py --tokenizer underthesea`: đo khả năng tìm các tài liệu **đã biết positive** từ toàn bộ 887 tài liệu, **không** gọi là Recall đầy đủ.
- `english_morphology_experiment.py`: stemming/lemmatization POS-aware dành cho tiếng Anh, không áp dụng Porter/WordNet cho tiếng Việt.

Không dùng CSConDa/EduCoQA trong thực hành này.

In [12]:
from IPython.display import display
crawl_summary = OUT_DIR/'crawl_eval'/'metric_summary.csv'
morph_summary = OUT_DIR/'29_english_morphology_POS_examples.csv'
if crawl_summary.exists():
    metrics=pd.read_csv(crawl_summary)
    baselines=metrics[metrics.strategy.eq('baseline')].set_index('method')
    print('SO SÁNH BASELINE (50 documents, 40 queries, 2.000 nhãn AI sơ bộ)')
    display(baselines[['P_at_5','Recall_at_5','Hit_at_1','MRR_at_5','nDCG_at_5']].round(4))
    print('Diễn giải: Precision@5 = trung bình số relevant trong Top-5 / 5; Recall@5 = phần relevant tìm lại được trên toàn tập 50 tài liệu đã có nhãn.')
    print('Recall cao không có nghĩa hệ thống xử lý hoàn hảo trên 887 documents; tập đánh giá 50 documents khá nhỏ và queries được thiết kế từ corpus.')
else:
    print('Chưa có P@5/Recall@5; cần đánh giá có nhãn từ corpus crawl.')
if morph_summary.exists():
    print('Thí nghiệm bổ sung POS-aware tiếng Anh:')
    display(pd.read_csv(morph_summary).head(12))
else:
    print('Chưa sinh bảng thí nghiệm POS-aware bổ sung; xem demo stemming/lemma đã lưu tại Bước 5, có thể chạy english_morphology_experiment.py.')


SO SÁNH BASELINE (50 documents, 40 queries, 2.000 nhãn AI sơ bộ)


,P_at_5,Recall_at_5,Hit_at_1,MRR_at_5,nDCG_at_5
method,,,,,
bm25,0.230,0.9917,0.975,0.9875,0.9849
tfidf,0.225,0.9833,0.925,0.9500,0.9539


Diễn giải: Precision@5 = trung bình số relevant trong Top-5 / 5; Recall@5 = phần relevant tìm lại được trên toàn tập 50 tài liệu đã có nhãn.
Recall cao không có nghĩa hệ thống xử lý hoàn hảo trên 887 documents; tập đánh giá 50 documents khá nhỏ và queries được thiết kế từ corpus.
Chưa sinh bảng thí nghiệm POS-aware bổ sung; xem demo stemming/lemma đã lưu tại Bước 5, có thể chạy english_morphology_experiment.py.


### Kiểm tra thực tế trên toàn bộ 887 documents (partial-label diagnostic)

Đây là **kiểm tra bổ sung** giúp thấy mức độ khó tăng khi corpus lớn. Dữ liệu ngoài danh sách positive đã biết là *unjudged*, **không được tính thành negative**.

In [13]:
from IPython.display import display
full_887 = OUT_DIR / 'crawl_eval' / 'full_887_diagnostic_summary.csv'
if full_887.is_file():
    diag=pd.read_csv(full_887)
    print('KNOWN-POSITIVE RETRIEVAL TRÊN 887 DOCUMENTS (KHÔNG PHẢI Recall@5 ĐẦY ĐỦ)')
    display(diag.round(4))
    print('Bởi vì chỉ biết các positive trong tập 50, tài liệu khác trong 887 phải xem là UNKNOWN, không tự gán negative.')
    print('AnyKnownPositive@5 đo tỷ lệ query tìm được ít nhất 1 positive đã biết trong 5 kết quả đầu.')
else:
    print('Cần chạy: python evaluate_887_diagnostic.py --tokenizer underthesea')


KNOWN-POSITIVE RETRIEVAL TRÊN 887 DOCUMENTS (KHÔNG PHẢI Recall@5 ĐẦY ĐỦ)


,strategy,method,AnyKnownPositive_at_1,AnyKnownPositive_at_5,AnyKnownPositive_at_10,KnownPositiveCoverage_at_5,mean_first_known_positive_rank,queries_scored,oov_queries
0,baseline,bm25,0.650,0.950,0.975,0.9292,1.900,40,0
1,baseline,tfidf,0.650,0.925,0.950,0.8792,2.375,40,0
2,lemmatized,bm25,0.600,0.975,0.975,0.9542,1.900,40,0
3,lemmatized,tfidf,0.625,0.900,0.950,0.8542,2.875,40,0
4,no_stopwords,bm25,0.600,0.975,0.975,0.9542,1.900,40,0
5,no_stopwords,tfidf,0.625,0.900,0.950,0.8542,2.875,40,0
6,stemmed,bm25,0.600,0.975,0.975,0.9542,1.900,40,0
7,stemmed,tfidf,0.625,0.900,0.950,0.8542,2.875,40,0


Bởi vì chỉ biết các positive trong tập 50, tài liệu khác trong 887 phải xem là UNKNOWN, không tự gán negative.
AnyKnownPositive@5 đo tỷ lệ query tìm được ít nhất 1 positive đã biết trong 5 kết quả đầu.


## Bước 9. Discussion & Interpretation (theo 8 chủ đề của đề)

1. **Stopwords:** Không bảo đảm loại stopwords luôn nâng kết quả. Corpus từ **93.829 xuống 78.046 tokens (giảm 16,82%)**, nhưng đây là giảm số token, không đồng nghĩa retrieval tốt hơn. Cố ý giữ từ phủ định như *không*, *chưa*.
2. **Stemming và Lemmatization:** Stemming cắt hậu tố theo quy tắc; lemmatization dựa vào lemma và từ loại (POS). Porter/WordNet là công cụ tiếng Anh. Ở corpus tiếng Việt, ba biến thể *no_stopwords*, *stemmed*, *lemmatized* có **cùng vocabulary 3.143**, không được nhận là đã triển khai đầy đủ morphology tiếng Việt.
3. **Thứ hạng Top-K:** TF-IDF dùng vector trọng số từ vựng và cosine; BM25 dùng term-frequency bão hòa cùng chuẩn hóa theo chiều dài văn bản. Điểm thô khác thang đo, nên chỉ so ranking và metrics.
4. **BM25 và độ dài:** `b=0.75` điều tiết chuẩn hóa độ dài, `k1=1.5` điều tiết bão hòa tần suất từ. Tài liệu dài không tự động được cộng điểm chỉ vì chứa nhiều từ.
5. **Kết quả P@5/Recall@5:** Trên **40 queries × 50 documents**, BM25 baseline đạt **P@5=0,230; R@5≈0,992**; TF-IDF baseline đạt **0,225; 0,983**. Số liệu có nhãn **AI hỗ trợ, chưa được kiểm chứng độc lập**, và chỉ áp dụng cho tập 50.
6. **Thêm Hit@1, MRR, nDCG:** Hit@1 quan tâm đúng ngay vị trí đầu; MRR thưởng cho vị trí positive đầu tiên cao; nDCG coi trọng thứ tự các tài liệu relevant. BM25 baseline có **Hit@1=0,975** so với TF-IDF **0,925**, cho thấy khác biệt mà P@5 gần bão hòa che khuất.
7. **Mở rộng lên 887 documents:** BM25 baseline tìm được ít nhất một positive đã biết trong Top-5 ở **95%** câu hỏi; TF-IDF ở **92,5%**. Đây là *known-positive diagnostic*, **không phải Recall@5 trên toàn 887**, vì nhãn bên ngoài 50 chưa đầy đủ.
8. **Kết luận về dữ liệu và giới hạn:** SQLite ban đầu cung cấp **1.022 passages**, corpus lab còn **889 documents** sau cleaning/chunking/deduplication, và **887 documents** được chọn làm auxiliary corpus. Có **2 bản ghi HCMUS chờ thẩm định**. Dataset crawl không được tự nhận là labeled query–positive training data. Query được tạo từ tài liệu đã chọn có thể khiến bài đánh giá dễ hơn truy vấn độc lập.

**Các tài liệu kiểm chứng thêm khi giảng viên yêu cầu:** `core/source_cleaning.py` (lọc nhiễu), `core/retrieval_lab.py` (tokenizer/TF-IDF/BM25), `crawl_eval/qrels_REVIEWED.csv` (nhãn AI sơ bộ), `crawl_eval/metric_summary.csv` (P/R), `crawl_eval/full_887_diagnostic_summary.csv` (887), `REPORT_SEG301_COMPLETE.md` (báo cáo). Không cần mở cả 20+ file CSV để trình bày.


## Kết luận: dataset nghiên cứu và giới hạn xác minh

- `04_crawled_auxiliary_corpus.csv`: corpus đầy đủ cho thực hành SEG301, bao gồm đoạn cần người kiểm tra.
- **`24_auxiliary_corpus_RESEARCH_READY.csv`**: tập chất lượng cao đã qua bộ kiểm tra tự động, **loại tạm các document có cảnh báo chưa được duyệt**. Đây là file khuyến nghị đưa vào mining pool FNAC-HNM.
- `25_pending_manual_review.csv`: dữ liệu chưa chắc chắn; có thể xem/xét giữ sau khi kiểm chứng nguồn.
- `26_research_release_manifest.json`: ghi số lượng, kiểm định hai bảng, và SHA256 để đảm bảo phiên bản dữ liệu.

Corpus crawl **không có nhãn relevance**. Không dùng làm TEST, không mặc định từng tài liệu crawl là negative chắc chắn. Hai mã ngành 7440107/7440107_DKD khác nhau giữa hai trang được cách ly chờ xác minh nguồn; kiểm tra cắt token bằng model vẫn là giới hạn cần ghi rõ trong báo cáo.

## Kiểm định cuối: dataset FNAC-HNM và chỉ mục lab

Tập lab 889 documents có các chỉ mục `models/`. Tập 887 documents là **auxiliary corpus không nhãn** phục vụ hard-negative mining; chỉ mục cho OURS-WEB phải dựng lại trên VIRE TRAIN + auxiliary, nên không lưu `models_research/`.


In [14]:
from core.research_release import sha256
ready_path = OUT_DIR/'24_auxiliary_corpus_RESEARCH_READY.csv'
ready = pd.read_csv(ready_path, keep_default_na=False)
pending = pd.read_csv(OUT_DIR/'25_pending_manual_review.csv', keep_default_na=False)
release_manifest = json.loads((OUT_DIR/'26_research_release_manifest.json').read_text(encoding='utf-8'))
full = pd.read_csv(OUT_DIR/'04_crawled_auxiliary_corpus.csv', keep_default_na=False)
assert ready.doc_id.is_unique and full.doc_id.is_unique
assert set(ready.doc_id).issubset(set(full.doc_id))
assert set(ready.doc_id).isdisjoint(set(pending.doc_id))
assert len(full) == len(ready) + len(pending)
assert sha256(ready_path) == release_manifest['research_ready_sha256']
for st in ('baseline','no_stopwords','stemmed','lemmatized'):
    assert sparse.load_npz(OUT_DIR/'models'/f'tfidf_{st}.npz').shape[0] == len(full)
print('Lab documents:', len(full), '| FNAC-HNM auxiliary:',len(ready),'| Pending:',len(pending))
print('Research CSV SHA-256:', sha256(ready_path))
print('Mining index: build from VIRE TRAIN + auxiliary when training (no standalone models_research).')
display(ready.groupby('source_site').size().rename('documents').reset_index())
display(pending[['doc_id','title','url']])
print('AUXILIARY DATASET, LAB INDEX AND QUARANTINE: VERIFIED')


Lab documents: 889 | FNAC-HNM auxiliary: 887 | Pending: 2
Research CSV SHA-256: 614b2ed5bd6901826b35781f329c6976c78804cf872bd5a70bae14a4b30c077a
Mining index: build from VIRE TRAIN + auxiliary when training (no standalone models_research).


,source_site,documents
0,fpt_telecom_support,88
1,hcmus_admissions,393
2,mobifone_5g_faq,12
3,momo_help,384
4,uel_admissions_faq,10


,doc_id,title,url
0,web_628ba69ce21a49cd96fb,Công nghệ Vật lý điện tử và tin học (Chương tr...,https://tuyensinh.hcmus.edu.vn/2026-phuong-huo...
1,web_17e93f5964dc6082cd41,Công nghệ Vật lý điện tử và tin học (Chương tr...,https://tuyensinh.hcmus.edu.vn/chi-tieu/


AUXILIARY DATASET, LAB INDEX AND QUARANTINE: VERIFIED
